# Point Clouds
## The fundamental input of any TDA pipeline

A **point cloud** is a finite metric space $(X, d)$ — the raw input for any TDA analysis. The choice of metric $d$ is not restricted to Euclidean distance; correlation-based or custom metrics are equally valid, which makes TDA particularly flexible for financial applications.

In this notebook we construct three point clouds with distinct topological signatures and recover their homological features via **persistent homology**.

## 1. Random Point Cloud

A set of 50 points drawn uniformly at random from $[0,1]^2$. No underlying topological structure — this serves as our **null case** against which persistent features are measured.

Note: random configurations can produce short-lived H1 generators due to accidental loop-like arrangements. These are distinguished from real features by their proximity to the diagonal in the persistence diagram.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Create a simple 2D point cloud
np.random.seed(42)
X = np.random.rand(50, 2)

# Visualize it
plt.figure(figsize=(6, 6))
plt.scatter(X[:, 0], X[:, 1], c='steelblue', s=50)
plt.title("Random Point Cloud")
plt.axis('equal')
plt.show()

## 2. Noisy Circle

100 points sampled from $S^1$ with additive Gaussian noise $\mathcal{N}(0, 0.1)$. The underlying space is homotopy equivalent to a circle, so we expect:

- **H0 = 1** — one connected component
- **H1 = 1** — one persistent loop generator

In [ ]:
# Create a noisy circle
theta = np.linspace(0, 2 * np.pi, 100)
X_circle = np.column_stack([np.cos(theta), np.sin(theta)])
X_circle += np.random.normal(0, 0.1, X_circle.shape)

# Visualize it
plt.figure(figsize=(6, 6))
plt.scatter(X_circle[:, 0], X_circle[:, 1], c='steelblue', s=50)
plt.title("Noisy Circle")
plt.axis('equal')
plt.show()

## 3. Two Clusters

Two Gaussian clouds centered at $(0,0)$ and $(3,3)$ with $\sigma = 0.2$. The underlying space has two connected components and no higher homology, so we expect:

- **H0 = 2** — two persistent connected components
- **H1 = 0** — no loops

In [ ]:
# Create two clusters
cluster1 = np.random.normal(loc=[0, 0], scale=0.2, size=(50, 2))
cluster2 = np.random.normal(loc=[3, 3], scale=0.2, size=(50, 2))
X_clusters = np.vstack([cluster1, cluster2])

# Visualize
plt.figure(figsize=(6, 6))
plt.scatter(X_clusters[:, 0], X_clusters[:, 1], c='steelblue', s=50)
plt.title("Two Clusters")
plt.axis('equal')
plt.show()

## Persistent Homology

Given a filtration $VR(X, \varepsilon_1) \subseteq VR(X, \varepsilon_2) \subseteq \cdots$, persistent homology tracks the birth and death of homological features across scales.

Each feature is represented as a pair $(b, d)$ in the **persistence diagram**:
- $b$ — the scale at which the feature appears
- $d$ — the scale at which it disappears
- **Persistence** $= d - b$ — features with high persistence are topologically significant
- Features near the diagonal ($d \approx b$) are considered noise

In [ ]:
from ripser import ripser
from persim import plot_diagrams

# Compute persistent homology on the noisy circle
diagrams = ripser(X_circle)['dgms']

# Visualize
plot_diagrams(diagrams, show=True)

In [ ]:
# Compute persistent homology on the two clusters
diagrams_clusters = ripser(X_clusters)['dgms']

# Visualize
plot_diagrams(diagrams_clusters, show=True)

In [ ]:
# Compute persistent homology on the random cloud
diagrams_random = ripser(X)['dgms']

# Visualize
plot_diagrams(diagrams_random, show=True)

## Comparison

| Space | H0 | H1 |
|-------|----|----|
| Random cloud | 1 persistent generator | No significant generators |
| Noisy circle $S^1$ | 1 persistent generator | 1 persistent generator |
| Two clusters | 2 persistent generators | No generators |

The persistence diagrams confirm that homology correctly recovers the underlying topological type of each space — even in the presence of noise.

In [ ]:
# Compare all three point clouds side by side
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Random cloud
axes[0].set_title("Random Cloud")
plot_diagrams(diagrams_random, ax=axes[0])

# Noisy circle
axes[1].set_title("Noisy Circle")
plot_diagrams(diagrams, ax=axes[1])

# Two clusters
axes[2].set_title("Two Clusters")
plot_diagrams(diagrams_clusters, ax=axes[2])

plt.tight_layout()
plt.show()

## Conclusions

Persistent homology recovers the correct homological signature of each space:

- **Random cloud** — no significant features, confirming the absence of underlying structure
- **Noisy circle** — one persistent H1 generator, correctly identifying $H_1(S^1) = \mathbb{Z}$
- **Two clusters** — two persistent H0 generators, correctly identifying two path components

This robustness to noise is the key property that makes TDA suitable for real financial data, where the underlying structure is obscured by market noise.